#### Position-Wise Feed-Forward Network (FFN):

##### Main layers of a transformer with Position-Wise FNN:

![transformer with Position-Wise FNN](image-22.png)

Position-wise Feed-Forward Network is an **expand-and-contract network** that transforms each sequence using the same dense layers.

![FFN Transform Layer](image-23.png)

In a Transformer, the Position-Wise Feed-Forward Network (FFN) acts like a dedicated **"mini-processor"** for every single word (or token) in your sequence. While the Attention layer looks at how words relate to each other, the FFN **focuses on refining the information** within each individual word.

##### Why is it called "Position-Wise"?

*“position-wise” network because it “transforms the representation at all the sequence positions using the same MLP & the exact same network is applied to each position independently."* 

- If you have a sentence of 10 words, the FFN runs 10 times—once for each word—using the same weights.
- There is no interaction between words inside this specific layer; that part was already handled by the Attention layer.

The Position-Wise Feed-Forward Network (FFN) consists of two fully connected dense layers, or a multi-layer perceptron (MLP). The hidden layer, which is known as ***d_ffn***, is generally set to a value about four times that of ***d_model***. This is why it is sometimes known as an expand-and-contract network.

FNN has a size of (d_model, d_ffn) in its first layer, which means it has to be broadcast across each sequence during tensor multiplication. This means each sequence is multiplied by the same weights. If identical sequences are input, the outputs will also be identical. This same logic applies to the second dense layer of size (d_ffn, d_model), which returns the tensor to its original size.

![Feed-Forward Neural Network (FNN) layer](image-20.png)

The text in the image describes the components and behavior of a Position-Wise Feed-Forward Network:
- First Dense Layer: Expands input dimension from $d_{model}\rightarrow d_{ffn}$ .
- ReLU Activation: Introduces non-linearity, prevents vanishing gradients.
- Second Dense Layer: Projects back to original size $d_{ffn}\rightarrow d_{model}$ .
- Broadcasting: Same weights applied across all sequence positions.

Qus: why called expansion-extractio?

***"Expansion provides the "thinking space" needed to process the context gathered by attention into meaningful knowledge."***

The expansion from $d_{model}$  to $d_{ffn}$  (usually a 4x increase, like 512 to 2048) is often called the "expansion-extraction" phase. It’s necessary for a few key reasons:

- Increasing "Capacity" for Knowledge
    - A Transformer’s internal state ($d_{model}$ ) is relatively small to keep the attention calculations efficient. By expanding to a larger dimension ($d_{ffn}$ ), you give the model more "neurons" to store and process complex patterns. It’s like moving from a narrow notebook to a large whiteboard to work out a difficult math problem before writing the final answer back in the notebook.
- Making Non-Linearity Effective
    - The ReLU activation function works by zeroing out negative values.If you didn't expand the dimension first, applying ReLU would likely destroy too much information.
    - By expanding to a higher dimension, the model can project data into a space where features are more "linearly separable," allowing it to filter out noise while keeping the essential signals intact.
- Key-Value Memory Analogy
    - Some researchers view the FFN as a **knowledge retrieval system**:
    - The First Layer ($d_{model} \to d_{ffn}$ ): Acts like "keys." It detects specific patterns or concepts in the input.
    - The Second Layer ($d_{ffn} \to d_{model}$ ): Acts like "values." Once a pattern is detected, this layer provides the corresponding information to update the token's representation.


#### Shape In Position-Wise Feed-Forward Network (FFN):
Let shape of output of tokenize -> embedding (Let d_model = 8) -> multi head attention laye ->  (3,6,8)
i.e. There are 3 sequences of 6 tokens with 8 dimensional embeddings.

Shaspe of hidden size $d_{ffn}$

Typical transformer setups use:

$d_{ffn}=4\times d_{model}$ 
That’s why if $d_{model} = 8, d_{ffn} = 32$ . But you could use $16, 64$ , or even $128$  depending on:

- Available memory and compute budget
- Desired model capacity
- Overfitting risk

Come to orignal problem, The new shape will be  first dense layer (3, 6, 8) dot (8, 32) → (3, 6, 32).
Then, the tensor can be passed through the second dense layer to return to its normal size, (3, 6, 32) x (32, 8) = (3, 6, 8). The values have changed according to the weights and activation function.

#### Internal Structure of Position-Wise Feed-Forward Network (FFN)

The FFN usually consists of two "linear" transformations with a non-linear "activation" function squeezed in the middle. Mathematically, it looks like this:

$FFN(x)=\text{max}(0,xW_{1}+b_{1})W_{2}+b_{2}$ 

- Step 1 (Expand): The first layer typically projects the input into a much higher-dimensional space (often 4x larger). This gives the model "room" to represent complex patterns.
- Step 2 (Activate): A non-linear function (like ReLU or GELU) is applied. This is the "complexity" mentioned in your image—without this, the model would just be a giant, simple linear calculator.
- Step 3 (Contract): The second layer projects the data back down to its original size so it can be passed to the next part of the Transformer.

The input to the model, X, will have a size of (batch_size, seq_length, d_model). The input will therefore go through the following transformations:
- $(batch\_size, seq\_length, d\_model) \times (d\_model, d\_ffn) = (batch\_size, seq\_length, d\_ffn)$ 
- $\max(0, (batch\_size, seq\_length, d\_ffn)) = (batch\_size, seq\_length, d\_ffn)$ 
- $(batch\_size, seq\_length, d\_ffn) \times (d\_ffn, d\_model) = (batch\_size, seq\_length, d\_model)$ 

#### Main Job Position-Wise Feed-Forward Network (FFN)

If the Attention layer is about context (gathering info from neighbors), the FFN layer is about processing that context. It takes the insights gathered by the attention layer and transforms them into a more useful, abstract representation for the final prediction.

# ENCODERS & DECODERS

![ENCODERS & DECODERS](image-24.png)

There are three main categories of transformers: encoder-only models, decoder-only models, and encoder-decoder models
- Different models will use different pieces of the transformer architecture

#### Encoder-Only Models:

**"to take an input sequence—like a sentence—and "understand" it by turning it into a set of context-rich numerical vectors."**

Only use the left side of the architecture, aka the encoder The encoder takes raw text and encodes it as an embedding representation of the text In short, it understands text.
- Application: Sentiment Analysis
- Input: “I love cold lemonade!”
- Output: Positive

Example: "The bat flew out of the cave."

The Final Encoder Output The Encoder outputs a "map" of vectors  where:
- "The" contains information about the subject following it.
- "bat" is enriched with "flying/animal/cave" context.
- "flew" contains info that the action was done by the bat.

#### Decoder-Only Models:

***"While the Encoder focuses on understanding the input, the Decoder focuses on generating the output (like a translation). Think of them as a Reader and a Writer."***

Only use the right side of the architecture, aka the decoder The decoder takes an input text sequence and infers* the next word In short, it generates text Application: Text Generation
- Input: “I love cold lemonade!”
- Output: “It’s the perfect drink! ”

#### Key Differences — Encoder vs Decoder

| Feature     | Encoder (The Reader) | Decoder (The Writer) |
|--------------|----------------------|----------------------|
| **Goal**     | Extract meaning and context. | Predict the next word in a sequence. |
| **Attention** | Uses **Self-Attention** (can see all words at once). | Uses **Masked Self-Attention** (can only see past words, not the future). |
| **Connection** | Operates independently on the input. | Constantly "listens" to the Encoder's output via **Encoder–Decoder Attention**. |

![Decoder Shifted Right Output](image-25.png)

The phrase "Outputs (shifted right)" is one of the most important parts of how a Transformer is trained. It essentially means that the Decoder is fed the correct answer, but delayed by one position.

1. ""shift"" creates the "Prompt" for the next word, Decoder is an auto-regressive model, meaning it predicts the next word based on the previous ones. To predict the first word of a translation, the model needs a starting point. We shift the actual target sentence to the right and insert a <START> token at the beginning. Without the shift, the model would be looking at the word it is currently trying to predict, which would be "cheating."

Example: Training on "I love AI"

If the target sentence is "I love AI", the Decoder doesn't get that exact sequence as input. It gets the "shifted" version:

| Step | Input to Decoder (Shifted) | Target (What it should predict) |
|------|-----------------------------|---------------------------------|
| 1    | <START>                     | I                               |
| 2    | <START> I                   | love                            |
| 3    | <START> I love              | AI                              |
| 4    | <START> I love AI           | ~END~                           |

Because we have the whole "shifted" sentence ready during training, the Transformer can process **all words at once** instead of waiting for one word to finish before starting the next (like older RNN models did).The **Masked Attention** we talked about earlier ensures that even though the whole shifted sentence is there, the model can only "see" the words to the left of the one it's currently predicting.

#### Encoder-Decoder Models

Use the entire architecture, both the encoder and decoder sides 

The encoder-decoder takes two inputs:
1. A text sequence
2. A shifted target sequence

Both are encoded as embeddings and combined to infer the next word In short, ***it understands and generates text***

- Application: Translation
    - Input: “I love cold lemonade!”
    - Output: “¡Me encanta la limonada fría!”


# Pretrained Large Language Models (LLMs)

LLMs use different "parts" of the Transformer architecture depending on their intended purpose:
- **Encoder-only (e.g., BERT, DistilBERT)**: These focus on "understanding" text. They are best for tasks like sentiment analysis or classifying text because they look at the whole sentence at once.
- **Encoder-Decoder (e.g., T5, BART)**: These are "translators." They take an input (Encoder) and generate a different output (Decoder). They excel at summarization and language translation.
- **Decoder-only (e.g., GPT)**: These are "generators." They are designed to predict the next word in a sequence, which makes them the gold standard for chatbots and creative writing.

Google: Created BERT and T5. 
Meta (Facebook): Created BART. 
OpenAI: Created the GPT series. 
Hugging Face: Created DistilBERT (a smaller, faster version of BERT).

#### Create new Conda environment ,activate and install packeg:
- list all Conda environments on your system:
    - conda env list
    - conda info --envs
- Check version
    - python --version
- Create a new environment named "nlp_transformer" with Python <version>
    - conda create --name nlp_transformer python=<version>
- Verify Python version inside the environment
    - python --version
- Make sure you're inside your new environment
    - conda activate myenv
- Install core scientific stack
    - conda install pandas numpy scipy scikit-learn jupyter pytorch torchvision torchaudio cpuonly -c pytorch -y
- Install Keras (via pip, since Conda often lags behind)    
    - pip install keras
    - pip install transformers
    - pip install tensorflow
- Verify Installation
    - python -c "import torch, transformers, keras, sklearn, scipy, pandas, numpy; print('All good!')"
    - conda list
- Method to install packeg in inside your Conda environment i.e. ""
    - Conda environments on your system
        - conda env list Or
        - conda info --envs
    - activate your environment
        - conda activate myenv
    - install nltk via conda-forge or Alternatively, if you prefer pip inside the Conda environment
        - pip install nltk
- nltk install supported packeg:
    - import nltk
    - nltk.download("stopwords")
    - nltk.download("punkt")
    - nltk.download("wordnet")
    - nltk.download("averaged_perceptron_tagger")
    - nltk.download("omw-1.4")
    - nltk.download("maxent_ne_chunker")
    - nltk.download("words")
- Install spaCy
    - pip install spacy
    - install the small English model package
        - python -m spacy download en_core_web_sm
    - medium, includes word vectors
        - python -m spacy download en_core_web_md
    - large, more accurate but heavier
        - python -m spacy download en_core_web_lg

In [ ]:
from transformers import pipeline
sentiment_analyzer = pipeline("sentiment-analysis",
                              model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
                              device=-1)
text = 'When life gives you lemons, make lemonade! 🙂'
sentiment_analyzer(text)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.996239423751831}]

In [2]:
text = "I fuck you mother fucker!!!"
print(sentiment_analyzer(text))
text = "I want sex with you very hard and brutally."
print(sentiment_analyzer(text))
text = "I want sex with you"
print(sentiment_analyzer(text))
text = "I want sex with you very hard"
print(sentiment_analyzer(text))
text = "Ram ja rha hai 🥶"
print(sentiment_analyzer(text))

[{'label': 'NEGATIVE', 'score': 0.9943853616714478}]
[{'label': 'POSITIVE', 'score': 0.5853140354156494}]
[{'label': 'POSITIVE', 'score': 0.9894071221351624}]
[{'label': 'POSITIVE', 'score': 0.7357922196388245}]
[{'label': 'POSITIVE', 'score': 0.5761924386024475}]


In [12]:
import pandas as pd
import os
import re
import string
import spacy

from nltk.corpus import stopwords

# create full path
file_path = os.path.join(os.getcwd(),'all-data.csv')
pd.set_option('display.max_colwidth',None)
df = pd.read_csv(file_path,encoding="ISO-8859-1")
df.columns = ['lable','Text']
print(df.head())

# text cleaning function
def clean_text(text):
    # lowercase
    text = text.lower()
    # remove punctuation
    text = text.translate(str.maketrans("", "", string.punctuation))
    # remove digits
    text = re.sub(r"\d+", "", text)
    # remove extra whitespace
    text = " ".join(text.split())
    # remove stopwords
    stop_words = set(stopwords.words("english"))
    text = " ".join([word for word in text.split() if word not in stop_words])
    return text

# apply cleaning
df["nltk_clen_txt"] = df["Text"].apply(clean_text)

# load spaCy English model
nlp = spacy.load("en_core_web_sm")

# text cleaning function using spaCy
def clean_text_spacy(text):
    # lowercase
    text = text.lower()
    # remove punctuation
    text = text.translate(str.maketrans("", "", string.punctuation))
    # remove digits
    text = re.sub(r"\d+", "", text)
    # remove extra whitespace
    text = " ".join(text.split())
    # remove stopwords with spaCy
    doc = nlp(text)
    text = " ".join([token.lemma_ for token in doc if not token.is_stop])
    return text
#print(df)
# apply cleaning
df["spacy_cln_txt"] = df["Text"].apply(clean_text_spacy)
df.columns

      lable  \
0   neutral   
1  negative   
2  positive   
3  positive   
4  positive   

                                                                                                                                                                                                                                   Text  
0                                        Technopolis plans to develop in stages an area of no less than 100,000 square meters in order to host companies working in computer technologies and telecommunications , the statement said .  
1  The international electronic industry company Elcoteq has laid off tens of employees from its Tallinn facility ; contrary to earlier layoffs the company contracted the ranks of its office workers , the daily Postimees reported .  
2                        With the new production plant the company would increase its capacity to meet the expected increase in demand and would improve the use of raw materials and therefore increase the pr

Index(['lable', 'Text', 'nltk_clen_txt', 'spacy_cln_txt'], dtype='str')

#### pipeline API details
1. task (str, optional):
    - The task you want to run (e.g., "sentiment-analysis", "text-classification", "translation", "summarization", "question-answering", "feature-extraction", "automatic-speech-recognition", "image-classification", etc.). If omitted, the task is inferred from the model.
2. model (str or PreTrainedModel, optional):
    - Model name from Hugging Face Hub (e.g., "distilbert-base-uncased-finetuned-sst-2-english") or a local model path. Can also pass a model object already loaded.
3. tokenizer (str or PreTrainedTokenizer, optional):
    - Tokenizer name/path or object. If not provided, inferred from the model. feature_extractor (str or PreTrained
4. FeatureExtractor, optional). 
    - Used for audio/image pipelines (e.g., Wav2Vec2, ViT).
5. processor (str or PreTrainedProcessor, optional):
    - For multimodal tasks (e.g., vision+text, speech+text).

In [7]:
from transformers import pipeline, logging

# hide all non-critical warnings
logging.set_verbosity_error()

# create a sentiment analyzer
sentiment_analyzer = pipeline(task ="sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
    #device="mps", # speed up the query by using our GPU if available
    device=-1,
    truncation=True)

# apply the sentiment analyzer to a series
sentiment_scores = df.spacy_cln_txt.apply(sentiment_analyzer)
sentiment_scores

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

0       [{'label': 'NEGATIVE', 'score': 0.897343814373...
1       [{'label': 'NEGATIVE', 'score': 0.994283258914...
2       [{'label': 'POSITIVE', 'score': 0.991213023662...
3       [{'label': 'NEGATIVE', 'score': 0.957163989543...
4       [{'label': 'NEGATIVE', 'score': 0.967526972293...
                              ...                        
4840    [{'label': 'NEGATIVE', 'score': 0.999272286891...
4841    [{'label': 'NEGATIVE', 'score': 0.994193971157...
4842    [{'label': 'NEGATIVE', 'score': 0.985368788242...
4843    [{'label': 'NEGATIVE', 'score': 0.992645621299...
4844    [{'label': 'NEGATIVE', 'score': 0.993094801902...
Name: spacy_cln_txt, Length: 4845, dtype: object